# Gold: tablas para las preguntas de negocio
Cada tabla de `dunnhumby.gold` responde una de las 4 preguntas de negocio, a partir de las tablas de `dunnhumby.silver`.

In [0]:
# SECCIÓN 0 · Funciones comunes de Gold
from pyspark.sql import functions as F
from pyspark.sql.window import Window

def guardar_gold(df, nombre):
    """Agrega _fecha_proceso, guarda la tabla en gold y muestra cuántas filas tiene."""
    df = df.withColumn("_fecha_proceso", F.current_timestamp())
    (df.write.format("delta")
       .mode("overwrite")
       .option("overwriteSchema", "true")   # permite re-ejecutar aunque cambie el esquema
       .saveAsTable(f"dunnhumby.gold.{nombre}"))
    print(f"{nombre}: {spark.table(f'dunnhumby.gold.{nombre}').count():,} filas")

## 1. Pregunta 1: ¿Qué departamentos generan más ventas y cómo evolucionan en el tiempo?
Tabla `ventas_departamento_periodo`: ventas por departamento y periodo de 4 semanas, sin combustible ni misceláneos.

In [0]:
# SECCIÓN 1 · Pregunta 1 · ventas_departamento_periodo (decisiones D1, D2 y D3)
tx = spark.table("dunnhumby.silver.transaction_data")
pr = spark.table("dunnhumby.silver.product").select("product_id", "department", "es_no_mercaderia")

# Todo producto vendido debe existir en product; si no, el join interno perdería ventas
assert tx.join(pr, "product_id", "left_anti").count() == 0, "Hay productos vendidos que no existen en product"

periodo = F.ceil(F.col("week_no") / 4).cast("int")   # D3: periodos de 4 semanas

ventas = (
    tx.join(pr, "product_id", "inner")
      .filter(~F.col("es_no_mercaderia"))            # D2: sin combustible ni misceláneos
      .groupBy(periodo.alias("periodo"), "department")
      .agg(F.sum("sales_value").alias("ventas"),     # D1: la métrica de dinero es sales_value
           F.count("*").alias("lineas"),
           F.countDistinct("basket_id").alias("canastas"))
)

# El último periodo solo tiene 2 semanas; esta columna evita leer esa caída como una baja real
semanas = tx.groupBy(periodo.alias("periodo")).agg(F.countDistinct("week_no").alias("semanas_en_periodo"))

guardar_gold(ventas.join(semanas, "periodo", "left"), "ventas_departamento_periodo")

In [0]:
# SECCIÓN 1 · Verificación de ventas_departamento_periodo
# Esperado: ventas_total 7,382,582.79 y lineas_total 2,550,922 (mercadería de la exploración); 26 periodos
g1 = spark.table("dunnhumby.gold.ventas_departamento_periodo")
display(g1.agg(
    F.sum("ventas").alias("ventas_total"),
    F.sum("lineas").alias("lineas_total"),
    F.countDistinct("periodo").alias("periodos"),
    F.countDistinct("department").alias("departamentos"),
))

In [0]:
# SECCIÓN 1 · Respuesta 1a: ranking de departamentos por ventas totales
ranking = (
    g1.groupBy("department").agg(F.sum("ventas").alias("ventas"))
      .withColumn("pct_ventas", F.round(100 * F.col("ventas") / F.sum("ventas").over(Window.partitionBy()), 2))
      .orderBy(F.desc("ventas"))
)
display(ranking.limit(10))

In [0]:
# SECCIÓN 1 · Respuesta 1b: evolución de los 5 departamentos con más ventas (solo periodos completos de 4 semanas)
top5 = [r["department"] for r in ranking.limit(5).collect()]
display(
    g1.filter(F.col("department").isin(top5) & (F.col("semanas_en_periodo") == 4))
      .select("periodo", "department", "ventas")
      .orderBy("periodo")
)

Databricks visualization. Run in Databricks to view.

## 2. Pregunta 2: ¿Gastan más los hogares que canjean cupones que los que no?
Tabla `gasto_hogar_cupones`: una fila por hogar, con su gasto, los cupones canjeados y las campañas recibidas.

In [0]:
# SECCIÓN 2 · Pregunta 2 · gasto_hogar_cupones (decisiones D2b y D5)
tx = spark.table("dunnhumby.silver.transaction_data")
red = spark.table("dunnhumby.silver.coupon_redempt")
camp = spark.table("dunnhumby.silver.campaign_table")

# D2b: el gasto del hogar usa todas las ventas
gasto = tx.groupBy("household_key").agg(
    F.sum("sales_value").alias("ventas_total"),
    (-F.sum(F.col("coupon_disc") + F.col("coupon_match_disc"))).alias("descuento_cupon_total"),  # en Silver es negativo; aquí se guarda positivo
    F.countDistinct("basket_id").alias("n_canastas"),
)

canjes = red.groupBy("household_key").agg(F.count("*").alias("n_cupones_canjeados"))
campanas = camp.groupBy("household_key").agg(F.countDistinct("campaign").alias("n_campanas_recibidas"))

# LEFT JOIN: un hogar sin canjes ni campañas se queda con 0 en lugar de desaparecer
g2 = (gasto.join(canjes, "household_key", "left")
           .join(campanas, "household_key", "left")
           .fillna(0, subset=["n_cupones_canjeados", "n_campanas_recibidas"])
           .withColumn("canjeo_cupon", F.col("n_cupones_canjeados") > 0))

guardar_gold(g2, "gasto_hogar_cupones")

In [0]:
# SECCIÓN 2 · Verificación de gasto_hogar_cupones
# Esperado: 2,500 hogares; ventas_total 8,057,443.80 (todas las ventas de Silver); 2,318 cupones; 7,208 campañas; 434 hogares que canjearon
g2 = spark.table("dunnhumby.gold.gasto_hogar_cupones")
display(g2.agg(
    F.count("*").alias("hogares"),
    F.sum("ventas_total").alias("ventas_total"),
    F.sum("n_cupones_canjeados").alias("cupones_canjeados"),
    F.sum("n_campanas_recibidas").alias("campanas_recibidas"),
    F.sum(F.col("canjeo_cupon").cast("int")).alias("hogares_canjearon"),
))

In [0]:
# SECCIÓN 2 · Respuesta 2a: gasto de los hogares que canjearon cupones frente a los que no
display(
    g2.groupBy("canjeo_cupon").agg(
        F.count("*").alias("hogares"),
        F.round(F.avg("ventas_total"), 2).alias("gasto_promedio"),
        F.round(F.percentile_approx("ventas_total", 0.5), 2).alias("gasto_mediano"),   # la mediana es menos sensible a hogares muy grandes
        F.round(F.avg("n_canastas"), 1).alias("canastas_promedio"),
        F.round(F.sum("ventas_total") / F.sum("n_canastas"), 2).alias("gasto_por_canasta"),
    ).orderBy("canjeo_cupon")
)

In [0]:
# SECCIÓN 2 · Respuesta 2b: lo mismo, según si el hogar recibió alguna campaña
display(
    g2.groupBy((F.col("n_campanas_recibidas") > 0).alias("recibio_campana")).agg(
        F.count("*").alias("hogares"),
        F.round(F.avg("ventas_total"), 2).alias("gasto_promedio"),
        F.round(F.percentile_approx("ventas_total", 0.5), 2).alias("gasto_mediano"),
        F.round(F.avg("n_canastas"), 1).alias("canastas_promedio"),
        F.round(F.sum("ventas_total") / F.sum("n_canastas"), 2).alias("gasto_por_canasta"),
    ).orderBy("recibio_campana")
)

## 3. Pregunta 3: ¿Cómo cambia el gasto según el ingreso y la composición del hogar?
Tabla `gasto_por_perfil_hogar`: gasto por hogar según su ingreso y su composición, incluido el grupo de hogares sin datos demográficos.

In [0]:
# SECCIÓN 3 · Pregunta 3 · gasto_por_perfil_hogar (decisiones D2b y D4)
tx = spark.table("dunnhumby.silver.transaction_data")
hh = spark.table("dunnhumby.silver.hh_demographic").select("household_key", "income_desc", "income_orden", "hh_comp_desc")

gasto = tx.groupBy("household_key").agg(
    F.sum("sales_value").alias("ventas_total"),   # D2b: todas las ventas del hogar
    F.countDistinct("basket_id").alias("n_canastas"),
)

# D4: LEFT JOIN para no perder los hogares sin registro demográfico (income_desc no tiene nulos en Silver)
base = gasto.join(hh, "household_key", "left")
tiene_demografia = F.col("income_desc").isNotNull()

# Dimensión 1: ingreso. El grupo sin datos va al final (orden 99)
ingreso = base.select(
    F.lit("ingreso").alias("dimension"),
    F.coalesce(F.col("income_desc"), F.lit("Sin datos demográficos")).alias("categoria"),
    F.coalesce(F.col("income_orden"), F.lit(99)).cast("int").alias("orden"),
    "ventas_total", "n_canastas",
)

# Dimensión 2: composición del hogar. "Composición desconocida" (orden 90) es distinto de "Sin datos demográficos" (99)
ORDEN_COMP = ["Single Female", "Single Male", "1 Adult Kids", "2 Adults No Kids", "2 Adults Kids"]
mapa_comp = F.create_map(*[x for i, r in enumerate(ORDEN_COMP, 1) for x in (F.lit(r), F.lit(i))])
composicion = base.select(
    F.lit("composicion_hogar").alias("dimension"),
    F.when(~tiene_demografia, "Sin datos demográficos")
     .when(F.col("hh_comp_desc").isNull(), "Composición desconocida")
     .otherwise(F.col("hh_comp_desc")).alias("categoria"),
    F.when(~tiene_demografia, 99)
     .when(F.col("hh_comp_desc").isNull(), 90)
     .otherwise(mapa_comp[F.col("hh_comp_desc")]).cast("int").alias("orden"),
    "ventas_total", "n_canastas",
)

perfil = (
    ingreso.unionByName(composicion)
    .groupBy("dimension", "categoria", "orden")
    .agg(F.count("*").alias("n_hogares"),
         F.sum("ventas_total").alias("ventas_total"),
         F.round(F.avg("ventas_total"), 2).alias("gasto_promedio_hogar"),
         F.round(F.percentile_approx("ventas_total", 0.5), 2).alias("gasto_mediano_hogar"),
         F.round(F.avg("n_canastas"), 1).alias("canastas_promedio"),
         F.round(F.sum("ventas_total") / F.sum("n_canastas"), 2).alias("gasto_por_canasta"))
)

guardar_gold(perfil, "gasto_por_perfil_hogar")

In [0]:
# SECCIÓN 3 · Verificación de gasto_por_perfil_hogar
# Esperado: ingreso con 13 categorías y composicion_hogar con 7; en ambas 2,500 hogares y ventas_total 8,057,443.80
g3 = spark.table("dunnhumby.gold.gasto_por_perfil_hogar")
display(
    g3.groupBy("dimension").agg(
        F.count("*").alias("categorias"),
        F.sum("n_hogares").alias("hogares"),
        F.sum("ventas_total").alias("ventas_total"),
    )
)

In [0]:
# SECCIÓN 3 · Respuesta 3a: gasto por nivel de ingreso
display(
    g3.filter("dimension = 'ingreso'").orderBy("orden")
      .select("categoria", "n_hogares", "gasto_promedio_hogar", "gasto_mediano_hogar", "canastas_promedio", "gasto_por_canasta")
)

In [0]:
# SECCIÓN 3 · Respuesta 3b: gasto por composición del hogar
display(
    g3.filter("dimension = 'composicion_hogar'").orderBy("orden")
      .select("categoria", "n_hogares", "gasto_promedio_hogar", "gasto_mediano_hogar", "canastas_promedio", "gasto_por_canasta")
)

## 4. Pregunta 4: ¿Qué hogares concentran el gasto y cuáles dejaron de comprar?
Tabla `segmentos_hogar`: una fila por hogar, con su decil de gasto y su estado de actividad.

In [0]:
# SECCIÓN 4 · Pregunta 4 · segmentos_hogar (decisiones D2b y D6)
tx = spark.table("dunnhumby.silver.transaction_data")

semana_max = tx.agg(F.max("week_no")).first()[0]   # última semana del dataset (102)

hogares = tx.groupBy("household_key").agg(
    F.sum("sales_value").alias("ventas_total"),    # D2b: todas las ventas del hogar
    F.countDistinct("basket_id").alias("n_canastas"),
    F.max("week_no").alias("ultima_semana"),
).withColumn("semanas_sin_comprar", F.lit(semana_max) - F.col("ultima_semana"))

# D6: activo 0 a 3 semanas sin comprar, en riesgo 4 a 12, inactivo 13 o más
estado = (F.when(F.col("semanas_sin_comprar") <= 3, "Activo")
           .when(F.col("semanas_sin_comprar") <= 12, "En riesgo")
           .otherwise("Inactivo"))

# decil_gasto = 1 es el 10 % de hogares que más gasta; household_key desempata para que el resultado sea siempre el mismo
decil = F.ntile(10).over(Window.orderBy(F.desc("ventas_total"), "household_key"))

guardar_gold(
    hogares.withColumn("estado_actividad", estado).withColumn("decil_gasto", decil),
    "segmentos_hogar",
)

In [0]:
# SECCIÓN 4 · Verificación de segmentos_hogar
# Esperado: Activo 1,991, En riesgo 335, Inactivo 174 (2,500 en total, ventas 8,057,443.80); 250 hogares en cada decil
g4 = spark.table("dunnhumby.gold.segmentos_hogar")
display(
    g4.groupBy("estado_actividad")
      .agg(F.count("*").alias("hogares"), F.sum("ventas_total").alias("ventas_total"))
      .orderBy("estado_actividad")
)
display(g4.groupBy("decil_gasto").count().orderBy("decil_gasto"))

In [0]:
# SECCIÓN 4 · Respuesta 4a: concentración del gasto por decil de hogares
por_decil = g4.groupBy("decil_gasto").agg(
    F.count("*").alias("hogares"),
    F.sum("ventas_total").alias("ventas"),
    F.round(F.avg("ventas_total"), 2).alias("gasto_promedio"),
)
total = F.sum("ventas").over(Window.partitionBy())
display(
    por_decil
      .withColumn("pct_ventas", F.round(100 * F.col("ventas") / total, 2))
      .withColumn("pct_acumulado", F.round(100 * F.sum("ventas").over(Window.orderBy("decil_gasto")) / total, 2))
      .orderBy("decil_gasto")
)

In [0]:
# SECCIÓN 4 · Respuesta 4b: hogares y gasto histórico según su estado de actividad
por_estado = g4.groupBy("estado_actividad").agg(
    F.count("*").alias("hogares"),
    F.sum("ventas_total").alias("ventas"),
    F.round(F.avg("ventas_total"), 2).alias("gasto_promedio"),
    F.round(F.avg("n_canastas"), 1).alias("canastas_promedio"),
)
display(
    por_estado
      .withColumn("pct_hogares", F.round(100 * F.col("hogares") / F.sum("hogares").over(Window.partitionBy()), 1))
      .withColumn("pct_ventas", F.round(100 * F.col("ventas") / F.sum("ventas").over(Window.partitionBy()), 1))
      .orderBy("estado_actividad")
)

In [0]:
# SECCIÓN 4 · Respuesta 4c: estado de actividad dentro de cada decil de gasto
display(
    g4.groupBy("decil_gasto")
      .pivot("estado_actividad", ["Activo", "En riesgo", "Inactivo"])
      .count()
      .fillna(0)
      .orderBy("decil_gasto")
)